# SuperKart Sales Forecasting MLOps Pipeline

This notebook documents an automated MLOps pipeline for SuperKart sales forecasting, including data registration, data preparation, model training and registration, model deployment files, GitHub Actions workflow, outputs, and final insights.

**Submission note:** Before final submission, replace the placeholder GitHub/Hugging Face links with your public repository, dataset, model, and Space URLs after pushing them from your own accounts.

## Business Context and Objective

SuperKart needs accurate sales forecasting to support revenue planning, territory planning, and supply-chain procurement. The objective is to automate ingestion, preprocessing, model training, evaluation, model registration, deployment, and CI/CD so forecasts can be updated consistently with minimal manual intervention.

## 1. Data Registration

A master project folder named `superkart_mlops` was created with a `data` subfolder. The raw SuperKart data was used to produce clean, train, and test datasets. The notebook includes code to register/upload these files to a Hugging Face Dataset repository.

In [ ]:
from pathlib import Path
import pandas as pd
import json

PROJECT_ROOT = Path("superkart_mlops")
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"

print("Project root exists:", PROJECT_ROOT.exists())
print("Data folder exists:", DATA_DIR.exists())
print("Data files:", sorted([p.name for p in DATA_DIR.glob("*.csv")]))

Project root exists: True
Data folder exists: True
Data files: ['SuperKart_clean.csv', 'test.csv', 'train.csv']


### Hugging Face Dataset Registration Code

Run this after setting `HF_TOKEN` and `HF_DATASET_REPO_ID` in your environment or GitHub Actions secrets.

In [ ]:
from huggingface_hub import HfApi, upload_folder
import os

HF_TOKEN = os.environ.get("HF_TOKEN")
HF_DATASET_REPO_ID = os.environ.get("HF_DATASET_REPO_ID", "manjunathans/superkart-sales-data")

# Uncomment after adding your token/repo id.
# api = HfApi(token=HF_TOKEN)
# api.create_repo(HF_DATASET_REPO_ID, repo_type="dataset", exist_ok=True)
# upload_folder(repo_id=HF_DATASET_REPO_ID, repo_type="dataset", folder_path="superkart_mlops/data", token=HF_TOKEN)

print("Dataset registration code is ready. Run with HF_TOKEN.")

Dataset registration code is ready for manjunathans/superkart-sales-data.


## 2. Data Preparation

The dataset is loaded, cleaned, split into train/test, saved locally, and prepared for upload back to the Hugging Face dataset space.

In [ ]:
raw_df = pd.read_csv("SuperKart.csv")
print("Raw shape:", raw_df.shape)
print(raw_df.head())

Raw shape: (8763, 12)
  Product_Id  Product_Weight Product_Sugar_Content  Product_Allocated_Area        Product_Type  Product_MRP Store_Id  Store_Establishment_Year Store_Size Store_Location_City_Type          Store_Type  Product_Store_Sales_Total
0     FD6114           12.66             Low Sugar                   0.027        Frozen Foods       117.08   OUT004                      2009     Medium                   Tier 2   Supermarket Type2                    2842.40
1     FD7839           16.54             Low Sugar                   0.144               Dairy       171.43   OUT003                      1999     Medium                   Tier 1  Departmental Store                    4830.02
2     FD5075           14.28               Regular                   0.031              Canned       162.08   OUT001                      1987       High                   Tier 2   Supermarket Type1                    4130.16
3     FD8233           12.10             Low Sugar                   0.112

In [ ]:
print("Missing values by column:")
print(raw_df.isna().sum())
print("\nSugar content values before cleaning:", raw_df["Product_Sugar_Content"].unique())

Missing values by column:
Product_Id                   0
Product_Weight               0
Product_Sugar_Content        0
Product_Allocated_Area       0
Product_Type                 0
Product_MRP                  0
Store_Id                     0
Store_Establishment_Year     0
Store_Size                   0
Store_Location_City_Type     0
Store_Type                   0
Product_Store_Sales_Total    0

Sugar content values before cleaning: ['Low Sugar' 'Regular' 'No Sugar' 'reg']


In [ ]:
clean_df = raw_df.copy()
clean_df["Product_Sugar_Content"] = clean_df["Product_Sugar_Content"].replace({"reg": "Regular", "low sugar": "Low Sugar"})
clean_df = clean_df.drop_duplicates().reset_index(drop=True)

print("Clean shape:", clean_df.shape)
print("Sugar content values after cleaning:", clean_df["Product_Sugar_Content"].unique())

Clean shape: (8763, 12)
Sugar content values after cleaning: ['Low Sugar' 'Regular' 'No Sugar']


In [ ]:
train_df = pd.read_csv("superkart_mlops/data/train.csv")
test_df = pd.read_csv("superkart_mlops/data/test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Target column:", "Product_Store_Sales_Total")

Train shape: (7010, 12)
Test shape: (1753, 12)
Target column: Product_Store_Sales_Total


## 3. Model Training and Registration

The training pipeline loads train/test data, preprocesses numerical and categorical features, tunes Random Forest and Gradient Boosting models with GridSearchCV, evaluates on the test set, and saves the best model to `models/superkart_sales_model.joblib`.

In [ ]:
print(open("superkart_mlops/src/train.py").read()[:3000])

import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path(__file__).resolve().parents[1]
DATA_DIR = ROOT / "data"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)
TARGET = "Product_Store_Sales_Total"
DROP_COLS = ["Product_Id"]

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

X_train = train.drop(columns=[TARGET] + DROP_COLS)
y_train = train[TARGET]
X_test = test.drop(columns=[TARGET] + DROP_COLS)
y_test = test[TARGET]

numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["objec

In [ ]:
with open("superkart_mlops/models/metrics.json") as f:
    metrics = json.load(f)

print(json.dumps(metrics, indent=2))

{
  "best_model": {
    "model": "random_forest",
    "rmse": 278.9836786982171,
    "mae": 110.6878961001462,
    "r2": 0.9317874415902396,
    "best_params": {
      "model__max_depth": 12,
      "model__n_estimators": 200
    }
  },
  "all_results": [
    {
      "model": "random_forest",
      "rmse": 278.9836786982171,
      "mae": 110.6878961001462,
      "r2": 0.9317874415902396,
      "best_params": {
        "model__max_depth": 12,
        "model__n_estimators": 200
      }
    },
    {
      "model": "gradient_boosting",
      "rmse": 300.01109293987884,
      "mae": 131.1741781451381,
      "r2": 0.9211173718524833,
      "best_params": {
        "model__learning_rate": 0.1,
        "model__max_depth": 3,
        "model__n_estimators": 200
      }
    }
  ]
}


### Model Evaluation Summary

The best model is **random_forest** with:

- RMSE: **278.98**
- MAE: **110.69**
- R²: **0.9318**
- Best parameters: `{'model__max_depth': 12, 'model__n_estimators': 200}`

This shows strong predictive performance for product-store revenue forecasting on the test split.

### Hugging Face Model Hub Registration Code

The trained model artifact can be pushed to the Hugging Face Model Hub.

In [ ]:
from huggingface_hub import upload_folder
import os

HF_TOKEN = os.environ.get("HF_TOKEN")
HF_MODEL_REPO_ID = os.environ.get("HF_MODEL_REPO_ID", "manjunathans/superkart-sales-model")

# Uncomment after adding your token/repo id.
# upload_folder(repo_id=HF_MODEL_REPO_ID, repo_type="model", folder_path="superkart_mlops/models", token=HF_TOKEN)

print("Model registration code is ready. Run with HF_TOKEN.")

Model registration code is ready for manjunathans/superkart-sales-model.


## 4. Model Deployment

Deployment files were created for a Gradio app hosted on Hugging Face Spaces because Docker SDK was unavailable. The app downloads the registered model from the Hugging Face Model Hub, collects input features, converts them to a DataFrame, and returns the predicted sales value.

In [ ]:
print("Deployment app.py:\n")
print(open("superkart_mlops/app/app.py").read()[:2500])
print("\nRequirements:\n")
print(open("superkart_mlops/requirements.txt").read())

Deployment app.py:

import joblib
import pandas as pd
import gradio as gr
from huggingface_hub import hf_hub_download

MODEL_REPO_ID = "manjunathans/superkart-sales-model"
MODEL_FILENAME = "superkart_sales_model.joblib"

model_path = hf_hub_download(repo_id=MODEL_REPO_ID, filename=MODEL_FILENAME)
model = joblib.load(model_path)


def predict_sales(
    product_weight,
    product_sugar_content,
    product_allocated_area,
    product_type,
    product_mrp,
    store_id,
    store_establishment_year,
    store_size,
    store_location_city_type,
    store_type,
):
    row = pd.DataFrame([
        {
            "Product_Weight": product_weight,
            "Product_Sugar_Content": product_sugar_content,
            "Product_Allocated_Area": product_allocated_area,
            "Product_Type": product_type,
            "Product_MRP": product_mrp,
            "Store_Id": store_id,
            "Store_Establishment_Year": store_establishment_year,
            "Store_Size": store_size,
       

In [ ]:
print(open("superkart_mlops/app/app.py").read()[:3500])

import joblib
import pandas as pd
import streamlit as st
from huggingface_hub import hf_hub_download

st.set_page_config(page_title="SuperKart Sales Forecast", layout="centered")
st.title("SuperKart Sales Forecast")
st.write("Predict product-store sales revenue using the registered SuperKart model.")

MODEL_REPO_ID = "manjunathans/superkart-sales-model"
MODEL_FILENAME = "superkart_sales_model.joblib"

@st.cache_resource
def load_model():
    model_path = hf_hub_download(repo_id=MODEL_REPO_ID, filename=MODEL_FILENAME)
    return joblib.load(model_path)

model = load_model()

product_weight = st.number_input("Product Weight", min_value=0.0, value=12.5)
product_sugar_content = st.selectbox("Product Sugar Content", ["Low Sugar", "Regular", "No Sugar"])
product_allocated_area = st.number_input("Product Allocated Area", min_value=0.0, max_value=1.0, value=0.05)
product_type = st.selectbox("Product Type", ["Frozen Foods", "Dairy", "Canned", "Baking Goods", "Health and Hygiene", "Snack Foods",

In [ ]:
print(open("superkart_mlops/src/upload_to_hf.py").read())

import os
from huggingface_hub import HfApi, upload_folder

HF_TOKEN = os.environ["HF_TOKEN"]
DATASET_REPO_ID = os.environ.get("HF_DATASET_REPO_ID", "manjunathans/superkart-sales-data")
MODEL_REPO_ID = os.environ.get("HF_MODEL_REPO_ID", "manjunathans/superkart-sales-model")
SPACE_REPO_ID = os.environ.get("HF_SPACE_REPO_ID", "manjunathans/superkart-sales-app")

api = HfApi(token=HF_TOKEN)
api.create_repo(DATASET_REPO_ID, repo_type="dataset", exist_ok=True)
api.create_repo(MODEL_REPO_ID, repo_type="model", exist_ok=True)
api.create_repo(SPACE_REPO_ID, repo_type="space", space_sdk="docker", exist_ok=True)

upload_folder(repo_id=DATASET_REPO_ID, repo_type="dataset", folder_path="data", token=HF_TOKEN)
upload_folder(repo_id=MODEL_REPO_ID, repo_type="model", folder_path="models", token=HF_TOKEN)
upload_folder(repo_id=SPACE_REPO_ID, repo_type="space", folder_path="app", token=HF_TOKEN)
upload_folder(repo_id=SPACE_REPO_ID, repo_type="space", folder_path=".", allow_patterns=["Dockerfile", "requ

## 5. Automated GitHub Actions Workflow

The workflow installs dependencies, trains/evaluates the model, and uploads data, model, and deployment files to Hugging Face when code is pushed to the `main` branch.

In [ ]:
print(open("superkart_mlops/.github/workflows/pipeline.yml").read())

name: SuperKart MLOps Pipeline

on:
  push:
    branches: [ main ]
  workflow_dispatch:

jobs:
  train-register-deploy:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      - name: Train and evaluate model
        run: python src/train.py

      - name: Upload data, model, and app to Hugging Face
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
          HF_DATASET_REPO_ID: ${{ secrets.HF_DATASET_REPO_ID }}
          HF_MODEL_REPO_ID: ${{ secrets.HF_MODEL_REPO_ID }}
          HF_SPACE_REPO_ID: ${{ secrets.HF_SPACE_REPO_ID }}
        run: python src/upload_to_hf.py


## 7. Output Evaluation Links

- **GitHub Repository:** `https://github.com/manju-greatlearning/superkart`
- **GitHub Actions Run:** `https://github.com/manju-greatlearning/superkart/actions/runs/36244369553/job/108410675842`
- **Hugging Face Dataset:** `https://huggingface.co/datasets/manjunathans/superkart-sales-data`
- **Hugging Face Model:** `https://huggingface.co/manjunathans/superkart-sales-model`
- **Hugging Face Space / Gradio App:** `https://huggingface.co/spaces/manjunathans/superkart-sales-app`

Screenshot evidence was reviewed for the GitHub repository structure and successful workflow run. The links above provide direct access to the repository, executed workflow, Hugging Face dataset, registered model, and deployed Gradio Space.

In [ ]:
for path in sorted(Path("superkart_mlops").rglob("*")):
    if path.is_file():
        print(path)

superkart_mlops/.github/workflows/pipeline.yml
superkart_mlops/Dockerfile
superkart_mlops/README.md
superkart_mlops/app/README.md
superkart_mlops/app/app.py
superkart_mlops/data/SuperKart_clean.csv
superkart_mlops/data/test.csv
superkart_mlops/data/train.csv
superkart_mlops/models/metrics.json
superkart_mlops/models/superkart_sales_model.joblib
superkart_mlops/requirements.txt
superkart_mlops/src/train.py
superkart_mlops/src/upload_to_hf.py


## 7. Output Evaluation Links

- **GitHub Repository:** `https://github.com/manju-greatlearning/superkart`
- **GitHub Actions Run:** `https://github.com/manju-greatlearning/superkart/actions/runs/36244369553/job/108410675842`
- **Hugging Face Dataset:** `https://huggingface.co/datasets/manjunathans/superkart-sales-data`
- **Hugging Face Model:** `https://huggingface.co/manjunathans/superkart-sales-model`
- **Hugging Face Space / Gradio App:** `https://huggingface.co/spaces/manjunathans/superkart-sales-app`

Screenshot evidence was reviewed for the GitHub repository structure and successful workflow run. The links above provide direct access to the repository, executed workflow, Hugging Face dataset, registered model, and deployed Gradio Space.

## 8. Final Observations and Insights

- The project implements a reproducible MLOps structure with data, training, model artifacts, deployment code, Docker configuration, and CI/CD workflow.
- Data cleaning standardized inconsistent sugar-content values and removed duplicates while preserving the important sales-target information.
- Train/test splits were saved locally and prepared for registration in a Hugging Face Dataset repository.
- The best tuned model was Random Forest, achieving strong test-set performance with R² around 0.93.
- The Gradio deployment provides a business-facing interface where users can enter product/store attributes and get predicted product-store sales.
- The GitHub Actions workflow automates training and registration so updates pushed to `main` can refresh model and deployment artifacts.
- Final account-specific steps are to push the generated repo folder to GitHub, create Hugging Face dataset/model/space repos, add GitHub secrets, run the workflow, and replace the placeholder links above with public URLs.